<p style="text-align:center">
    <img src="https://media.licdn.com/dms/image/v2/D560BAQGRKQWaOldXvg/company-logo_200_200/company-logo_200_200/0/1723551442229/cambo_ai_logo?e=2147483647&v=beta&t=FrK5k676UJMjoOULl4R3euChg1VvClew28fxNscSd9E" width="80" alt="CamboAI Logo">
</p>

# **I. Objectives**

**Estimated time needed: 45 minutes**

**After completing this lab you will be able to:**

- The primary objective of this project is to extract, transform, and store product data, including nested attributes, from a APIs into a structured PostgreSQL database.
- This process involves creating a robust data pipeline that accurately extracts nested information, such as product ratings, and transforms it into a normalized format suitable for database storage.
- By achieving this, the project aims to facilitate efficient data retrieval, analysis, and management, enabling comprehensive insights into product details and their performance metrics for further data analytics and decision-making processes.

# **II. Import Libraries**

In [1]:
# install required libraries
%pip install requests
%pip install psycopg2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# import required libraries
import requests
import psycopg2
import pandas as pd

# **III. Extract Data from API and Convert into DataFrame**

In [3]:
# define the URL of the API endpoint
url = 'https://fakestoreapi.com/products/'

In [4]:
# send a GET request to the API
response = requests.get(url)

# check if the request was successful
if response.status_code == 200:
    # parse the json response
    data = response.json()
    # print the data
    print('Status is succeed - ', response.status_code)
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")

Status is succeed -  200


In [5]:
# convert data into DataFrame
df = pd.DataFrame(data)

In [6]:
# show data
df.head()

,id,title,price,description,category,image,rating
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,Your perfect pack for everyday use and walks i...,men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_S...,"{'rate': 3.9, 'count': 120}"
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,"Slim-fitting style, contrast raglan long sleev...",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_S...,"{'rate': 4.1, 'count': 259}"
2,3,Mens Cotton Jacket,55.99,great outerwear jackets for Spring/Autumn/Wint...,men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_U...,"{'rate': 4.7, 'count': 500}"
3,4,Mens Casual Slim Fit,15.99,The color could be slightly different between ...,men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_U...,"{'rate': 2.1, 'count': 430}"
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,"From our Legends Collection, the Naga was insp...",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_U...,"{'rate': 4.6, 'count': 400}"


In [7]:
# extract 'rating_rate' and 'rating_count' from the nested 'rating' data
df['rating_rate'] = df['rating'].apply(lambda x: x['rate'] if isinstance(x, dict) and 'rate' in x else None)
df['rating_count'] = df['rating'].apply(lambda x: x['count'] if isinstance(x, dict) and 'count' in x else None)

# **IV. Create connection to PostgreSQL DB**

In [16]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()
# database connection details
db_config = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD', ''),
    'database': os.getenv('POSTGRES_DATABASE')
    }

# **V. Create Target Table and Load Data**

In [17]:
# connect to PostgreSQL database and create table
try:
    conn = psycopg2.connect(**db_config)
    cursor = conn.cursor()

    # SQL statement to create the table
    create_table_query = '''
    CREATE TABLE IF NOT EXISTS tbl_products (
        id SERIAL PRIMARY KEY,
        title VARCHAR(255) NOT NULL,
        price NUMERIC(10, 2) NOT NULL,
        description TEXT,
        category VARCHAR(100),
        image VARCHAR(255),
        rating_rate NUMERIC(3, 1),
        rating_count INTEGER
    );
    '''
    cursor.execute(create_table_query)
    conn.commit()

    # insert data into the products table
    insert_query = '''
    INSERT INTO tbl_products (title, price, description, category, image, rating_rate, rating_count)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    '''
    for _, row in df.iterrows():
        cursor.execute(insert_query, (row['title'], row['price'], row['description'], row['category'], row['image'], row['rating_rate'], row['rating_count']))

    conn.commit()
    print("Data inserted successfully.")

except Exception as e:
    print("Error:", e)

finally:
    # close the connection
    if cursor:
        cursor.close()
    if conn:
        conn.close()

Data inserted successfully.


#Excellent! You have just completed this Notebook.

This notebook was created by [Chanthan DINA](https://www.linkedin.com/in/chanthandina/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!

## <h3 align="center"> &#169; CamboAI. All rights reserved. <h3/>